In [1]:
import os
import pandas as pd
from datetime import datetime
import json
import gc

folder_path_demanddetails = '/home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/'

# read active properties & needed columns
property_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/phagwara/eg_pt_property_phagwara_v2.csv',
    usecols=['id', 'propertyid', 'tenantid', 'createdtime', 'additionaldetails', 'ownershipcategory', 'status', 'usagecategory']
)
property_df = property_df[property_df['status'] == 'ACTIVE'].copy()

# read units
# unit_df = pd.read_csv(
#     '/home/prerna/Punjab/punjab-data-prod-analysis/srihargobindpur/eg_pt_unit.csv',
#     usecols=['propertyid', 'occupancytype']
# )



# read demand
demand_df = pd.read_csv(
    '/home/prerna/Punjab/punjab-data-prod-analysis/phagwara/eg_demand_phagwara.csv',
    dtype={"consumercode": str},
    low_memory=False,
    usecols=['id', 'taxperiodfrom', 'taxperiodto', 'consumercode', 'status', 'businessservice']
)
demand_df = demand_df[demand_df['status'] == 'ACTIVE'].copy()
demand_df = demand_df[demand_df['businessservice'] == 'PT'].copy()


# read demand details (memory‑efficient, in chunks)
all_chunks = []
needed_cols = ['demandid', 'taxamount', 'collectionamount', 'taxheadcode']
for filename in os.listdir(folder_path_demanddetails):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path_demanddetails, filename)
        print(f'Loading: {file_path}')
        chunk = pd.read_csv(file_path, usecols=needed_cols)
        all_chunks.append(chunk)
demand_details_df = pd.concat(all_chunks, ignore_index=True)
del all_chunks; gc.collect()

print("✅ Loaded data")

Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_4.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_34.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_38.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_6.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_60.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_44.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_17.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_22.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_59.csv
Loading: /home/prerna/Punjab/punjab-data-prod-analysis/phagwara/output_demand_detail/output_29.csv
Loading: /ho

In [2]:
print(len(property_df))         # number of rows in properties
# print(len(unit_df))             # number of rows in units
print(len(demand_df))   # number of rows in demand details
print(len(demand_details_df))   # number of rows in demand details

21958
142201
3248706


In [3]:
# join demand and demand details
joined_demand = demand_df.merge(demand_details_df, left_on='id', right_on='demandid', how='left', suffixes=('_demand', '_detail'))
print(joined_demand['id'].nunique())
del demand_details_df, demand_df; gc.collect()
joined_demand.head()

142201


,id,consumercode,businessservice,taxperiodfrom,taxperiodto,status,demandid,taxheadcode,taxamount,collectionamount
0,50c875db-c676-4214-abf4-907ebc2117d4,PT-1014-708136,PT,1396310400000,1427846399000,ACTIVE,50c875db-c676-4214-abf4-907ebc2117d4,PT_TIME_INTEREST,640.10,640.10
1,50c875db-c676-4214-abf4-907ebc2117d4,PT-1014-708136,PT,1396310400000,1427846399000,ACTIVE,50c875db-c676-4214-abf4-907ebc2117d4,PT_OWNER_EXEMPTION,0.00,0.00
2,50c875db-c676-4214-abf4-907ebc2117d4,PT-1014-708136,PT,1396310400000,1427846399000,ACTIVE,50c875db-c676-4214-abf4-907ebc2117d4,PT_CANCER_CESS,15.34,15.34
3,50c875db-c676-4214-abf4-907ebc2117d4,PT-1014-708136,PT,1396310400000,1427846399000,ACTIVE,50c875db-c676-4214-abf4-907ebc2117d4,PT_TAX,766.67,766.67
4,50c875db-c676-4214-abf4-907ebc2117d4,PT-1014-708136,PT,1396310400000,1427846399000,ACTIVE,50c875db-c676-4214-abf4-907ebc2117d4,PT_UNIT_USAGE_EXEMPTION,0.00,0.00


In [4]:
import pytz

# Correct: parse as datetime from milliseconds since epoch
joined_demand['taxperiodfrom'] = pd.to_datetime(joined_demand['taxperiodfrom'], unit='ms', utc=True)
joined_demand['taxperiodto'] = pd.to_datetime(joined_demand['taxperiodto'], unit='ms', utc=True)

# Convert to IST (Asia/Kolkata)
ist = pytz.timezone('Asia/Kolkata')
joined_demand['taxperiodfrom'] = joined_demand['taxperiodfrom'].dt.tz_convert(ist)
joined_demand['taxperiodto'] = joined_demand['taxperiodto'].dt.tz_convert(ist)

# Financial year calculation
def get_fy(date):
    if date.month >= 4:
        fy_start = date.year
        fy_end = date.year + 1
    else:
        fy_start = date.year - 1
        fy_end = date.year
    return f"{fy_start}-{str(fy_end)[-2:]}"

joined_demand['fy'] = joined_demand['taxperiodfrom'].apply(get_fy)

# Group by consumercode
result = joined_demand.groupby('consumercode')['fy'].agg(['min', 'max']).reset_index()
result.rename(columns={'min': 'earliest_fy', 'max': 'latest_fy'}, inplace=True)

print(result)

          consumercode earliest_fy latest_fy
0       PT-1014-052724     2018-19   2025-26
1       PT-1014-067025     2020-21   2022-23
2      PT-1014-1000000     2020-21   2025-26
3      PT-1014-1000004     2020-21   2025-26
4      PT-1014-1000006     2020-21   2025-26
...                ...         ...       ...
21778   PT-1014-999754     2020-21   2025-26
21779   PT-1014-999756     2020-21   2025-26
21780   PT-1014-999820     2020-21   2025-26
21781   PT-1014-999823     2020-21   2025-26
21782   PT-1014-999928     2020-21   2025-26

[21783 rows x 3 columns]


In [5]:
# Merge latest_fy onto joined_demand by consumercode
joined = joined_demand.merge(
    result[['consumercode', 'latest_fy']],
    on='consumercode',
    how='left'
)

# Filter only latest FY
latest_demand = joined[joined['fy'] == joined['latest_fy']]

# Pivot taxheadcode values into separate columns
pivoted = latest_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
# PT_TAX + PT_CANCER_CESS + PT_FIRE_CESS + PT_ROUNDOFF - (PT_OWNER_EXEMPTION + PT_UNIT_USAGE_EXEMPTION)
pivoted['latest_fy_taxamount'] = (
    pivoted.get('PT_TAX', 0) +
    pivoted.get('PT_CANCER_CESS', 0) +
    pivoted.get('PT_FIRE_CESS', 0) +
    pivoted.get('PT_ROUNDOFF', 0) -
    ( pivoted.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Merge back into result
result = result.merge(
    pivoted[['consumercode', 'latest_fy_taxamount']],
    on='consumercode',
    how='left'
)

print(result.head())


      consumercode earliest_fy latest_fy  latest_fy_taxamount
0   PT-1014-052724     2018-19   2025-26                 0.00
1   PT-1014-067025     2020-21   2022-23               421.65
2  PT-1014-1000000     2020-21   2025-26                 0.00
3  PT-1014-1000004     2020-21   2025-26               457.93
4  PT-1014-1000006     2020-21   2025-26              4151.47


In [6]:
# Calculating the tax amount (demand) of current year using formula
target_fy = "2025-26"
current_fy_demand = joined_demand[joined_demand['fy'] == target_fy]

# Pivot taxheadcode values into separate columns
pivoted_current = current_fy_demand.pivot_table(
    index='consumercode',
    columns='taxheadcode',
    values='taxamount',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Apply formula:
pivoted_current['current_fy_taxamount'] = (
    pivoted_current.get('PT_TAX', 0) +
    pivoted_current.get('PT_CANCER_CESS', 0) +
    pivoted_current.get('PT_FIRE_CESS', 0) +
    pivoted_current.get('PT_ROUNDOFF', 0) -
    ( pivoted_current.get('PT_OWNER_EXEMPTION', 0).abs() + pivoted_current.get('PT_UNIT_USAGE_EXEMPTION', 0).abs() )
)

# Keep only required cols
pivoted_current = pivoted_current[['consumercode', 'current_fy_taxamount']]

# Ensure all consumercodes are present
all_consumercodes = pd.DataFrame(joined_demand['consumercode'].unique(), columns=['consumercode'])
final = all_consumercodes.merge(pivoted_current, on='consumercode', how='left')
final['current_fy_taxamount'] = final['current_fy_taxamount'].fillna(0)

# Merge into result
result = result.merge(final, on='consumercode', how='left')
result['current_fy_taxamount'] = result['current_fy_taxamount'].fillna(0)

print(result.head())


      consumercode earliest_fy latest_fy  latest_fy_taxamount  \
0   PT-1014-052724     2018-19   2025-26                 0.00   
1   PT-1014-067025     2020-21   2022-23               421.65   
2  PT-1014-1000000     2020-21   2025-26                 0.00   
3  PT-1014-1000004     2020-21   2025-26               457.93   
4  PT-1014-1000006     2020-21   2025-26              4151.47   

   current_fy_taxamount  
0                  0.00  
1                  0.00  
2                  0.00  
3                457.93  
4               4151.47  


In [7]:
property_result_merged = property_df.merge(
    result,
    left_on='propertyid',
    right_on='consumercode',
    how='left'
)

print(property_result_merged)

                                         id       propertyid     tenantid  \
0      dcd1e947-407d-4dce-bd12-56a426636fec  PT-1014-1035693  pb.phagwara   
1      4c1fa5a9-230e-481b-b8ad-329804877095   PT-1014-881544  pb.phagwara   
2      e3e66501-3669-474a-830f-50bcdeafd065  PT-1014-2110512  pb.phagwara   
3      66acb675-bbf5-4a8f-babb-9eb2f9894994  PT-1014-2004760  pb.phagwara   
4      65cb9a53-ede1-4700-a9d9-d62b66b60320  PT-1014-2007961  pb.phagwara   
...                                     ...              ...          ...   
21953  55e61260-3253-4859-a822-d29bfd56c68d  PT-1014-1992525  pb.phagwara   
21954  63121883-2aea-4fa6-ac83-b1b33cc6158b  PT-1014-1010354  pb.phagwara   
21955  7cde651b-7b54-4d82-ba46-629ce5acf731   PT-1014-878378  pb.phagwara   
21956  9beb289c-39db-4c2a-876d-a9a5a88f76ca  PT-1014-1992726  pb.phagwara   
21957  6605127d-393e-44eb-b8e8-41fb5bade3bd  PT-1014-1989803  pb.phagwara   

       status          ownershipcategory                 usagecategory  \
0

In [8]:
property_result_merged.to_csv('Punjab_Data_Analysis_phagwara_final_2.csv', index=False)